# Notebook 02. Image-derived features with provenance

**OHDSI Global Symposium 2026. Tutorial "Bringing FAIR to Imaging Research with the Medical Imaging OMOP Extension"**

**Hands-on II (10:00–10:30)**

Notebook 01 turned DICOM **headers** into OMOP. This notebook opens the **pixels**, runs an open-source chest X-ray model, and stores what it finds so that anyone can later answer: *which image, which model, which version, when?*

| Wiki step | Section | DICOM2OMOP counterpart |
|---|---|---|
| 7. Do you have image-derived features? | 1 | — |
| 8. Is MI-CDM implemented? | 0 | — |
| 9. Standardized concepts for the features? | 2 | — |
| 11. Define custom concepts | 2 | not done here: gaps stay `0` with the source value kept (open item) |
| 10. ETL image features | 3–4 | `transform_load_imaging_algo_results.ipynb` |
| ETL guide §9. Data quality | 5 | — |

**What gets written** (ETL guide §6–7):

```
IMAGE_OCCURRENCE ─┐
                   ├─► IMAGE_FEATURE  (what was found, where, by which algorithm)
PERSON ───────────┘  │  image_feature_event_field_concept_id = 1147304 (OBSERVATION)
                           └─► OBSERVATION (the value: score, yes/no call, ratio, area)
```

> ⚠️ **Not a diagnosis.** The model outputs are research artifacts. Nothing here goes into `CONDITION_OCCURRENCE`.

> ⚠️ **Concept IDs are a teaching example.** Re-check every one in [Athena](https://athena.ohdsi.org) for your vocabulary version.

**GPU recommended:** `Runtime → Change runtime type → T4 GPU`. On CPU everything still works; it takes a few minutes longer.

## 0. Setup and wiki step 8 — is MI-CDM in place?

Image features must hang off an `IMAGE_OCCURRENCE` row (wiki step 8). The catch-up cell below checks for notebook 01's database and rebuilds it if this is a fresh runtime.

In [ ]:
%pip install -q "pydicom>=3.0" pylibjpeg pylibjpeg-libjpeg remotezip torchxrayvision scikit-image

In [ ]:
#@title ⏩ Catch-up — loads the shared code and rebuilds earlier notebooks' output if this runtime lacks it (run me)
import os, sys, importlib
sys.path.insert(0, os.getcwd())
MICDM_PART1_SRC = r'''"""micdm_part1.py — generated from Notebook 01. Used by the catch-up cells of notebooks 02/03.
Do not edit by hand: regenerate from the notebook builder."""


# Configuration. Re-run this cell after a runtime restart; every later cell relies on it.
import os, re, glob, json, sqlite3, datetime, warnings, urllib.request
import pandas as pd

warnings.filterwarnings("ignore", message="Invalid value for VR")   # RSNA stores PatientAge as a bare integer
pd.set_option("display.max_colwidth", 60)

BASE_DIR = os.environ.get(
    "MICDM_BASE",
    "/content/micdm_tutorial" if os.path.isdir("/content") else os.path.abspath("./micdm_tutorial"))
DB_PATH = os.path.join(BASE_DIR, "omop_micdm.db")
DICOM_DIR = os.path.join(BASE_DIR, "dicom")
VOCAB_DIR = os.path.join(BASE_DIR, "vocab")
for _d in (BASE_DIR, DICOM_DIR, VOCAB_DIR):
    os.makedirs(_d, exist_ok=True)

# ---- tutorial-scale knobs ------------------------------------------------------------
N_IMAGES = int(os.environ.get("MICDM_N_IMAGES", 100))   # keep 50–200 in the session
SUBSET_OFFSET = 0                                          # sorted file list -> same images for everyone
RSNA_ZIP_URL = ("https://s3.amazonaws.com/east1.public.rsna.org/AI/2018/"
                "pneumonia-challenge-dataset-adjudicated-kaggle_2018.zip")

# ---- DICOM vocabulary: DICOM2OMOP staging files, pinned to one commit for reproducibility
DICOM2OMOP_COMMIT = "56be2b80c9ba2c9b8f19dc36a8ab5919aed6fb88"
DICOM2OMOP_RAW = ("https://raw.githubusercontent.com/paulnagy/DICOM2OMOP/"
                  f"{DICOM2OMOP_COMMIT}/files/OMOP%20CDM%20Staging/")
VOCAB_FILES = {
    "concept": "omop_table_staging_v5.csv",           # DICOM attributes + CS value sets (2128xxxxxx)
    "maps_to_value": "cs_values_maps_to_value.csv",   # attribute  --Maps to value-->  value
    "maps_to": "cs_values_maps_to.csv",               # value      --Maps to-->        standard (SNOMED)
}
DICOM_CONCEPT_RANGE = (2128000000, 2128999999)

# ---- concepts fixed by OMOP / the ImageWG conventions ---------------------------------
CONCEPT_UNMAPPED = 0
CONCEPT_TYPE_EHR = 32817            # Type Concept 'EHR'
CONCEPT_CHEST_XRAY = 4163872        # SNOMED 399208008 'Plain X-ray of chest' (Procedure)
CONCEPT_MALE, CONCEPT_FEMALE = 8507, 8532
CONCEPT_UNIT_MM = 8588              # UCUM 'mm'
EVENT_FIELD_MEASUREMENT = 1147330   # ETL guide §7: image_feature -> MEASUREMENT

# meas_event_field_concept_id for rows linked to IMAGE_OCCURRENCE. The CDM vocabulary has no
# Field concept for image_occurrence.image_occurrence_id yet, so 0 ('No matching concept') is
# written until the ImageWG designates one. The link itself is measurement_event_id.
FIELD_IMAGE_OCCURRENCE_ID = CONCEPT_UNMAPPED

# ---- DICOM2OMOP attribute-selection rules (transform_imaging_metadata.ipynb) ------------
MEASURABLE_VRS = ["AT", "CS", "DA", "DT", "DS", "FL", "FD", "IS", "SL", "SS", "SV", "TM", "UL", "US", "UV"]
NUMERIC_VRS = ["DS", "FL", "FD", "IS", "SL", "SS", "SV", "UL", "US", "UV"]
ALWAYS_INCLUDE_TAGS = ["00080070"]       # Manufacturer (LO), kept by DICOM2OMOP for analysis
MAX_VALUE_LEN = 50
# ETL guide §4: attributes that populate IMAGE_OCCURRENCE (not repeated in MEASUREMENT)
CORE_TAGS = {"00080020": "Study Date", "00100020": "Patient ID", "00080060": "Modality",
             "00180015": "Body Part Examined", "0020000D": "Study Instance UID",
             "0020000E": "Series Instance UID"}
UNIT_BY_TAG = {"00280030": CONCEPT_UNIT_MM,   # Pixel Spacing
               "00181164": CONCEPT_UNIT_MM,   # Imager Pixel Spacing
               "00180050": CONCEPT_UNIT_MM,   # Slice Thickness
               "00181110": CONCEPT_UNIT_MM,   # Distance Source to Detector
               "00181111": CONCEPT_UNIT_MM}   # Distance Source to Patient


def connect(db_path=None):
    conn = sqlite3.connect(db_path or DB_PATH)
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def q(sql, params=()):
    """Run a read-only query and return a DataFrame."""
    with connect() as conn:
        return pd.read_sql_query(sql, conn, params=params)


CDM_DDL = """
CREATE TABLE IF NOT EXISTS concept (
    concept_id        INTEGER NOT NULL PRIMARY KEY,
    concept_name      VARCHAR(255) NOT NULL,
    domain_id         VARCHAR(20)  NOT NULL,
    vocabulary_id     VARCHAR(20)  NOT NULL,
    concept_class_id  VARCHAR(20)  NOT NULL,
    standard_concept  VARCHAR(1),
    concept_code      VARCHAR(50)  NOT NULL,
    valid_start_date  DATE NOT NULL,
    valid_end_date    DATE NOT NULL,
    invalid_reason    VARCHAR(1));

CREATE TABLE IF NOT EXISTS concept_relationship (
    concept_id_1      INTEGER NOT NULL,
    concept_id_2      INTEGER NOT NULL,
    relationship_id   VARCHAR(20) NOT NULL,
    valid_start_date  DATE NOT NULL,
    valid_end_date    DATE NOT NULL,
    invalid_reason    VARCHAR(1),
    PRIMARY KEY (concept_id_1, concept_id_2, relationship_id));

CREATE TABLE IF NOT EXISTS person (
    person_id                   INTEGER NOT NULL PRIMARY KEY,
    gender_concept_id           INTEGER NOT NULL,
    year_of_birth               INTEGER NOT NULL,
    month_of_birth              INTEGER,
    day_of_birth                INTEGER,
    birth_datetime              TIMESTAMP,
    race_concept_id             INTEGER NOT NULL,
    ethnicity_concept_id        INTEGER NOT NULL,
    location_id                 INTEGER,
    provider_id                 INTEGER,
    care_site_id                INTEGER,
    person_source_value         VARCHAR(50),
    gender_source_value         VARCHAR(50),
    gender_source_concept_id    INTEGER,
    race_source_value           VARCHAR(50),
    race_source_concept_id      INTEGER,
    ethnicity_source_value      VARCHAR(50),
    ethnicity_source_concept_id INTEGER);

CREATE TABLE IF NOT EXISTS observation_period (
    observation_period_id         INTEGER NOT NULL PRIMARY KEY,
    person_id                     INTEGER NOT NULL REFERENCES person (person_id),
    observation_period_start_date DATE NOT NULL,
    observation_period_end_date   DATE NOT NULL,
    period_type_concept_id        INTEGER NOT NULL);

CREATE TABLE IF NOT EXISTS procedure_occurrence (
    procedure_occurrence_id     INTEGER NOT NULL PRIMARY KEY,
    person_id                   INTEGER NOT NULL REFERENCES person (person_id),
    procedure_concept_id        INTEGER NOT NULL,
    procedure_date              DATE NOT NULL,
    procedure_datetime          TIMESTAMP,
    procedure_end_date          DATE,
    procedure_end_datetime      TIMESTAMP,
    procedure_type_concept_id   INTEGER NOT NULL,
    modifier_concept_id         INTEGER,
    quantity                    INTEGER,
    provider_id                 INTEGER,
    visit_occurrence_id         INTEGER,
    visit_detail_id             INTEGER,
    procedure_source_value      VARCHAR(50),
    procedure_source_concept_id INTEGER,
    modifier_source_value       VARCHAR(50));

CREATE TABLE IF NOT EXISTS measurement (
    measurement_id                INTEGER NOT NULL PRIMARY KEY,
    person_id                     INTEGER NOT NULL REFERENCES person (person_id),
    measurement_concept_id        INTEGER NOT NULL,
    measurement_date              DATE NOT NULL,
    measurement_datetime          TIMESTAMP,
    measurement_time              VARCHAR(10),
    measurement_type_concept_id   INTEGER NOT NULL,
    operator_concept_id           INTEGER,
    value_as_number               NUMERIC,
    value_as_concept_id           INTEGER,
    unit_concept_id               INTEGER,
    range_low                     NUMERIC,
    range_high                    NUMERIC,
    provider_id                   INTEGER,
    visit_occurrence_id           INTEGER,
    visit_detail_id               INTEGER,
    measurement_source_value      VARCHAR(50),
    measurement_source_concept_id INTEGER,
    unit_source_value             VARCHAR(50),
    unit_source_concept_id        INTEGER,
    value_source_value            VARCHAR(50),
    measurement_event_id          INTEGER,
    meas_event_field_concept_id   INTEGER);
CREATE INDEX IF NOT EXISTS idx_measurement_event ON measurement (measurement_event_id, meas_event_field_concept_id);
CREATE INDEX IF NOT EXISTS idx_measurement_concept ON measurement (measurement_concept_id);
"""


def create_cdm_tables(conn):
    conn.executescript(CDM_DDL)


MICDM_DDL = """
CREATE TABLE IF NOT EXISTS image_occurrence (
    image_occurrence_id       INTEGER NOT NULL PRIMARY KEY,
    person_id                 INTEGER NOT NULL REFERENCES person (person_id),
    procedure_occurrence_id   INTEGER NOT NULL REFERENCES procedure_occurrence (procedure_occurrence_id),
    visit_occurrence_id       INTEGER,
    anatomic_site_concept_id  INTEGER,
    wadors_uri                VARCHAR(10000),
    local_path                VARCHAR(10000),
    image_occurrence_date     DATE NOT NULL,
    image_study_uid           VARCHAR(10000) NOT NULL,
    image_series_uid          VARCHAR(10000) NOT NULL,
    modality_concept_id       INTEGER NOT NULL);

CREATE TABLE IF NOT EXISTS image_feature (
    image_feature_id                      INTEGER NOT NULL PRIMARY KEY,
    person_id                             INTEGER NOT NULL REFERENCES person (person_id),
    image_occurrence_id                   INTEGER NOT NULL REFERENCES image_occurrence (image_occurrence_id),
    image_feature_event_field_concept_id  INTEGER,
    image_feature_event_id                INTEGER,
    image_feature_concept_id              INTEGER NOT NULL,
    image_feature_type_concept_id         INTEGER NOT NULL,
    image_finding_concept_id              INTEGER,
    image_finding_id                      INTEGER,
    anatomic_site_concept_id              INTEGER,
    alg_system                            VARCHAR(10000),
    alg_datetime                          TIMESTAMP);
CREATE UNIQUE INDEX IF NOT EXISTS idx_image_occurrence_series ON image_occurrence (image_study_uid, image_series_uid);
CREATE INDEX IF NOT EXISTS idx_image_feature_image ON image_feature (image_occurrence_id);
"""

# Tables written by the ETL, parent -> child. Re-running a step clears that table and
# everything downstream of it, so foreign keys never dangle.
ETL_TABLE_ORDER = ["person", "observation_period", "procedure_occurrence",
                   "image_occurrence", "measurement", "image_feature"]


def create_micdm_tables(conn):
    conn.executescript(MICDM_DDL)


def clear_from(conn, table):
    """Delete `table` and every ETL table downstream of it (children first)."""
    downstream = ETL_TABLE_ORDER[ETL_TABLE_ORDER.index(table):]
    for t in reversed(downstream):
        if t == "observation_period" and table not in ("person", "observation_period"):
            continue                                   # not a child of procedure/image tables
        conn.execute(f"DELETE FROM {t}")
    return downstream


# Standard concepts the tutorial refers to. This is a *display subset* so queries can show
# names — it is not a substitute for loading the Athena vocabularies at your site.
STANDARD_CONCEPTS = [
    # concept_id, name, domain, vocabulary, class, standard, code
    (0, "No matching concept", "Metadata", "None", "Undefined", None, "No matching concept"),
    (8507, "MALE", "Gender", "Gender", "Gender", "S", "M"),
    (8532, "FEMALE", "Gender", "Gender", "Gender", "S", "F"),
    (32817, "EHR", "Type Concept", "Type Concept", "Type Concept", "S", "OMOP4976890"),
    (4163872, "Plain X-ray of chest", "Procedure", "SNOMED", "Procedure", "S", "399208008"),
    (4184181, "Thoracic cavity structure", "Spec Anatomic Site", "SNOMED", "Body Structure", "S", "43799004"),
    (4213162, "Lung structure", "Spec Anatomic Site", "SNOMED", "Body Structure", "S", "39607008"),
    (8588, "millimeter", "Unit", "UCUM", "Unit", "S", "mm"),
    (1147330, "measurement", "Metadata", "CDM", "Table", "S", "CDM386"),
]


def download_vocabulary(vocab_dir=None):
    vocab_dir = vocab_dir or VOCAB_DIR
    paths = {}
    for key, fname in VOCAB_FILES.items():
        dest = os.path.join(vocab_dir, fname)
        if not os.path.exists(dest):
            urllib.request.urlretrieve(DICOM2OMOP_RAW + fname, dest)
        paths[key] = dest
    return paths


def _yyyymmdd_to_iso(s):
    s = str(s)
    return f"{s[:4]}-{s[4:6]}-{s[6:8]}" if len(s) == 8 and s.isdigit() else s


def insert_concepts(conn, rows):
    conn.executemany(
        """INSERT OR REPLACE INTO concept (concept_id, concept_name, domain_id, vocabulary_id,
               concept_class_id, standard_concept, concept_code, valid_start_date,
               valid_end_date, invalid_reason)
           VALUES (?,?,?,?,?,?,?,'1970-01-01','2099-12-31',NULL)""", rows)


def load_dicom_vocabulary(conn, paths):
    concept = pd.read_csv(paths["concept"], dtype=str)
    concept = concept.where(concept.notna(), None)
    concept["valid_start_date"] = concept["valid_start_date"].map(_yyyymmdd_to_iso)
    concept["valid_end_date"] = concept["valid_end_date"].map(_yyyymmdd_to_iso)
    cols = ["concept_id", "concept_name", "domain_id", "vocabulary_id", "concept_class_id",
            "standard_concept", "concept_code", "valid_start_date", "valid_end_date", "invalid_reason"]
    conn.executemany(f"INSERT OR REPLACE INTO concept ({', '.join(cols)}) VALUES ({', '.join('?' * len(cols))})",
                     concept[cols].itertuples(index=False, name=None))

    rel = pd.concat([pd.read_csv(paths["maps_to_value"]), pd.read_csv(paths["maps_to"])])
    conn.execute("DELETE FROM concept_relationship WHERE concept_id_1 BETWEEN ? AND ?", DICOM_CONCEPT_RANGE)
    conn.executemany(
        """INSERT OR REPLACE INTO concept_relationship
           (concept_id_1, concept_id_2, relationship_id, valid_start_date, valid_end_date, invalid_reason)
           VALUES (?,?,?,'1970-01-01','2099-12-31',NULL)""",
        rel[["concept_id_1", "concept_id_2", "relationship_id"]].itertuples(index=False, name=None))
    insert_concepts(conn, STANDARD_CONCEPTS)
    return len(concept), len(rel)


def download_rsna_subset(dest=None, n_images=None, offset=None, url=RSNA_ZIP_URL):
    """Pull n_images .dcm members out of the remote ZIP. Skips the download if enough files exist."""
    dest, n_images = dest or DICOM_DIR, n_images or N_IMAGES
    offset = SUBSET_OFFSET if offset is None else offset
    existing = sorted(glob.glob(os.path.join(dest, "**", "*.dcm"), recursive=True))
    if len(existing) >= n_images:
        print(f"{len(existing)} DICOM files already present — download skipped")
        return existing[:n_images] if len(existing) == n_images else existing
    from remotezip import RemoteZip
    with RemoteZip(url) as z:
        names = sorted(n for n in z.namelist() if n.lower().endswith(".dcm"))
        print(f"archive holds {len(names):,} DICOM files; taking [{offset}:{offset + n_images}]")
        for i, name in enumerate(names[offset:offset + n_images], 1):
            if not os.path.exists(os.path.join(dest, name)):
                z.extract(name, dest)
            if i % 25 == 0:
                print(f"  {i}/{n_images}")
    return sorted(glob.glob(os.path.join(dest, "**", "*.dcm"), recursive=True))


def index_dicom_archive(dicom_dir=None):
    import pydicom
    rows = []
    for path in sorted(glob.glob(os.path.join(dicom_dir or DICOM_DIR, "**", "*.dcm"), recursive=True)):
        ds = pydicom.dcmread(path, stop_before_pixels=True,
                             specific_tags=["PatientID", "StudyInstanceUID", "SeriesInstanceUID",
                                            "SOPInstanceUID", "Modality"])
        rows.append({"local_path": os.path.abspath(path),
                     "patient_id": str(ds.get("PatientID", "")),
                     "study_uid": str(ds.get("StudyInstanceUID", "")),
                     "series_uid": str(ds.get("SeriesInstanceUID", "")),
                     "sop_uid": str(ds.get("SOPInstanceUID", "")),
                     "modality": str(ds.get("Modality", "")),
                     "file_bytes": os.path.getsize(path)})
    return pd.DataFrame(rows)


SKIP_VRS = {"SQ", "OB", "OW", "OF", "OD", "OL", "OV", "UN", "DT"}

def flatten_header(ds):
    """Top-level DICOM elements -> list of (Tag, keyword, vr, vm, Value)."""
    out = []
    for elem in ds:
        if elem.VR in SKIP_VRS or elem.tag == 0x7FE00010:
            continue
        v = elem.value
        if v is None or v == "":
            value = None
        elif elem.VM > 1:
            value = ", ".join(str(x) for x in v)
        else:
            value = str(v)
        out.append((f"{elem.tag.group:04X}{elem.tag.element:04X}", elem.keyword, elem.VR, elem.VM, value))
    return out


def extract_series_metadata(inventory):
    import pydicom
    first = inventory.sort_values("local_path").groupby(["study_uid", "series_uid"], as_index=False).first()
    rows = []
    for rec in first.itertuples(index=False):
        ds = pydicom.dcmread(rec.local_path, stop_before_pixels=True)
        for tag, keyword, vr, vm, value in flatten_header(ds):
            rows.append((rec.study_uid, rec.series_uid, tag, keyword, vr, vm, value))
    return pd.DataFrame(rows, columns=["StudyUID", "SeriesUID", "Tag", "keyword", "vr", "vm", "Value"])


def map_tags_to_concepts(flat_df, conn):
    attrs = pd.read_sql_query(
        """SELECT concept_id, concept_code, concept_name FROM concept
           WHERE vocabulary_id = 'DICOM' AND concept_class_id = 'DICOM Attributes'""", conn)
    m = flat_df.merge(attrs, how="left", left_on="Tag", right_on="concept_code")
    m["concept_id"] = m["concept_id"].astype("Int64")
    return m


def select_attributes(metadata_omop):
    structured = metadata_omop["vr"].isin(MEASURABLE_VRS)
    too_long = metadata_omop["Value"].fillna("").str.len() > MAX_VALUE_LEN
    exclude = metadata_omop.loc[metadata_omop["concept_id"].notna() & structured & too_long, "Tag"].unique()
    keep = (metadata_omop["concept_id"].notna()
            & metadata_omop["Value"].notna()
            & ~metadata_omop["Tag"].isin(exclude)
            & (structured | metadata_omop["Tag"].isin(ALWAYS_INCLUDE_TAGS))
            & ~metadata_omop["Tag"].isin(list(CORE_TAGS)))
    return metadata_omop[keep].copy()


def dicom_age_years(raw):
    """DICOM AS ('051Y', '006M') or RSNA's bare integer -> years (float) or None."""
    m = re.match(r"^\s*(\d+)\s*([DWMY])?\s*$", str(raw or ""), re.IGNORECASE)
    if not m:
        return None
    n, unit = int(m.group(1)), (m.group(2) or "Y").upper()
    return {"Y": n, "M": n / 12, "W": n / 52.18, "D": n / 365.25}[unit]


def dicom_date_iso(raw):
    s = str(raw or "").strip()
    return f"{s[:4]}-{s[4:6]}-{s[6:8]}" if len(s) == 8 and s.isdigit() else None


def series_wide(conn, tags):
    """Pivot selected tags of stg_dicom_metadata_long to one row per series (DICOM2OMOP cell 23)."""
    long = pd.read_sql_query("SELECT StudyUID, SeriesUID, Tag, Value FROM stg_dicom_metadata_long", conn)
    wide = (long[long["Tag"].isin(tags)]
            .pivot(index=["StudyUID", "SeriesUID"], columns="Tag", values="Value").reset_index())
    wide.columns.name = None
    for t in tags:
        if t not in wide:
            wide[t] = None
    return wide


def etl_person(conn):
    clear_from(conn, "person")
    w = series_wide(conn, ["00100020", "00100040", "00101010", "00080020"])
    w = w.sort_values(["00100020", "00080020"])
    pts = w.groupby("00100020", as_index=False).first()          # first study per patient
    pts["person_id"] = range(1, len(pts) + 1)
    pts["gender_concept_id"] = pts["00100040"].str.upper().map(
        {"M": CONCEPT_MALE, "F": CONCEPT_FEMALE}).fillna(CONCEPT_UNMAPPED).astype(int)
    study_year = pts["00080020"].map(dicom_date_iso).str[:4].astype(float).fillna(1900)
    age = pts["00101010"].map(dicom_age_years)
    pts["year_of_birth"] = (study_year - age.fillna(0)).astype(int)

    conn.executemany(
        """INSERT INTO person (person_id, gender_concept_id, year_of_birth, race_concept_id,
               ethnicity_concept_id, gender_source_value) VALUES (?,?,?,0,0,?)""",
        pts[["person_id", "gender_concept_id", "year_of_birth", "00100040"]].itertuples(index=False, name=None))

    conn.execute("DROP TABLE IF EXISTS registry_idmap")
    conn.execute("CREATE TABLE registry_idmap (source_id VARCHAR(250) NOT NULL, person_id INTEGER NOT NULL, source_name VARCHAR(250))")
    conn.executemany("INSERT INTO registry_idmap VALUES (?,?,'RSNA DICOM PatientID')",
                     pts[["00100020", "person_id"]].itertuples(index=False, name=None))

    # observation period = span of imaging (the only data we have for these persons)
    conn.execute("""
        INSERT INTO observation_period (observation_period_id, person_id, observation_period_start_date,
                                        observation_period_end_date, period_type_concept_id)
        SELECT r.person_id, r.person_id, MIN(d.study_date), MAX(d.study_date), ?
        FROM registry_idmap r
        JOIN (SELECT l1.Value AS patient_id,
                     substr(l2.Value,1,4)||'-'||substr(l2.Value,5,2)||'-'||substr(l2.Value,7,2) AS study_date
              FROM stg_dicom_metadata_long l1
              JOIN stg_dicom_metadata_long l2 ON l2.SeriesUID = l1.SeriesUID AND l2.Tag = '00080020'
              WHERE l1.Tag = '00100020') d ON d.patient_id = r.source_id
        GROUP BY r.person_id""", (CONCEPT_TYPE_EHR,))
    return len(pts)


def etl_procedure_occurrence(conn):
    clear_from(conn, "procedure_occurrence")
    w = series_wide(conn, ["00100020", "00080020", "00080060", "00180015"])
    st = w.sort_values("SeriesUID").groupby("StudyUID", as_index=False).first().sort_values("StudyUID")
    ids = pd.read_sql_query("SELECT source_id, person_id FROM registry_idmap", conn)
    st = st.merge(ids, left_on="00100020", right_on="source_id", how="left")
    st["procedure_occurrence_id"] = range(1, len(st) + 1)
    is_cxr = st["00080060"].isin(["CR", "DX"]) & st["00180015"].str.upper().isin(["CHEST", "THORAX"])
    st["procedure_concept_id"] = is_cxr.map({True: CONCEPT_CHEST_XRAY, False: CONCEPT_UNMAPPED})
    st["procedure_date"] = st["00080020"].map(dicom_date_iso)
    st["procedure_source_value"] = (st["00080060"].fillna("") + " " + st["00180015"].fillna("")).str.strip()
    conn.executemany(
        """INSERT INTO procedure_occurrence (procedure_occurrence_id, person_id, procedure_concept_id,
               procedure_date, procedure_type_concept_id, procedure_source_value)
           VALUES (?,?,?,?,?,?)""",
        [(r.procedure_occurrence_id, r.person_id, r.procedure_concept_id, r.procedure_date,
          CONCEPT_TYPE_EHR, r.procedure_source_value) for r in st.itertuples()])
    st[["StudyUID", "procedure_occurrence_id"]].rename(columns={"StudyUID": "study_uid"}).to_sql(
        "stg_study_procedure", conn, if_exists="replace", index=False)
    if (~is_cxr).any():
        print(f"  ! {(~is_cxr).sum()} studies are not chest radiographs -> procedure_concept_id = 0")
    return len(st)


def value_concept_lookup(conn):
    """{(attribute_concept_id, value_code): value_concept_id} via 'Maps to value'."""
    df = pd.read_sql_query(
        """SELECT r.concept_id_1 AS attr_id, v.concept_code AS code, v.concept_id AS value_id
           FROM concept_relationship r JOIN concept v ON v.concept_id = r.concept_id_2
           WHERE r.relationship_id = 'Maps to value'""", conn)
    return {(int(a), str(c).upper()): int(v) for a, c, v in df.itertuples(index=False)}


def maps_to_lookup(conn):
    df = pd.read_sql_query("SELECT concept_id_1, concept_id_2 FROM concept_relationship WHERE relationship_id = 'Maps to'", conn)
    return dict(zip(df.concept_id_1.astype(int), df.concept_id_2.astype(int)))


def attribute_id(conn, tag):
    return int(conn.execute("SELECT concept_id FROM concept WHERE vocabulary_id='DICOM' AND concept_code=?", (tag,)).fetchone()[0])


def normalize_body_part(raw):
    return re.sub(r"[^A-Z]", "", str(raw or "").upper())


def etl_image_occurrence(conn):
    clear_from(conn, "image_occurrence")
    vmap, mapsto = value_concept_lookup(conn), maps_to_lookup(conn)
    attr_modality, attr_bodypart = attribute_id(conn, "00080060"), attribute_id(conn, "00180015")

    w = series_wide(conn, ["00100020", "00080020", "00080060", "00180015"]).sort_values(["StudyUID", "SeriesUID"])
    ids = pd.read_sql_query("SELECT source_id, person_id FROM registry_idmap", conn)
    proc = pd.read_sql_query("SELECT study_uid, procedure_occurrence_id FROM stg_study_procedure", conn)
    path = pd.read_sql_query(
        "SELECT series_uid, MIN(local_path) AS local_path FROM stg_dicom_inventory GROUP BY series_uid", conn)
    w = (w.merge(ids, left_on="00100020", right_on="source_id", how="left")
          .merge(proc, left_on="StudyUID", right_on="study_uid", how="left")
          .merge(path, left_on="SeriesUID", right_on="series_uid", how="left"))

    w["image_occurrence_id"] = range(1, len(w) + 1)
    w["modality_concept_id"] = [vmap.get((attr_modality, str(m).upper()), CONCEPT_UNMAPPED) for m in w["00080060"]]
    bp_value = [vmap.get((attr_bodypart, normalize_body_part(b))) for b in w["00180015"]]
    w["anatomic_site_concept_id"] = [mapsto.get(v, CONCEPT_UNMAPPED) if v else CONCEPT_UNMAPPED for v in bp_value]
    w["image_occurrence_date"] = w["00080020"].map(dicom_date_iso)

    conn.executemany(
        """INSERT INTO image_occurrence (image_occurrence_id, person_id, procedure_occurrence_id,
               visit_occurrence_id, anatomic_site_concept_id, wadors_uri, local_path,
               image_occurrence_date, image_study_uid, image_series_uid, modality_concept_id)
           VALUES (?,?,?,NULL,?,NULL,?,?,?,?,?)""",
        [(r.image_occurrence_id, r.person_id, r.procedure_occurrence_id, r.anatomic_site_concept_id,
          r.local_path, r.image_occurrence_date, r.StudyUID, r.SeriesUID, r.modality_concept_id)
         for r in w.itertuples()])
    unmapped = w.loc[w["modality_concept_id"] == 0, "00080060"].unique().tolist()
    if unmapped:
        print("  ! unmapped Modality values:", unmapped)
    return len(w)


def to_number(s):
    try:
        return float(s)
    except (TypeError, ValueError):
        return None


def etl_dicom_measurements(conn, selected):
    clear_from(conn, "measurement")
    io = pd.read_sql_query("SELECT image_occurrence_id, person_id, image_series_uid, image_occurrence_date FROM image_occurrence", conn)
    df = selected.merge(io, left_on="SeriesUID", right_on="image_series_uid", how="inner")

    # split multi-valued elements: one row per value (ETL guide §5.2)
    df["component"] = [v.split(", ") if vm > 1 else [v] for v, vm in zip(df["Value"], df["vm"])]
    df = df.explode("component").reset_index(drop=True)

    vmap = value_concept_lookup(conn)

    rows, mid = [], 1
    for r in df.itertuples():
        attr = int(r.concept_id)
        num = to_number(r.component) if r.vr in NUMERIC_VRS else None
        val_concept = vmap.get((attr, str(r.component).upper()), CONCEPT_UNMAPPED) if r.vr == "CS" else None
        rows.append((mid, int(r.person_id), attr, r.image_occurrence_date, CONCEPT_TYPE_EHR, num, val_concept,
                     UNIT_BY_TAG.get(r.Tag) if num is not None else None,
                     str(r.Value)[:50], attr, str(r.component)[:50],
                     int(r.image_occurrence_id), FIELD_IMAGE_OCCURRENCE_ID))
        mid += 1
    conn.executemany(
        """INSERT INTO measurement (measurement_id, person_id, measurement_concept_id, measurement_date,
               measurement_type_concept_id, value_as_number, value_as_concept_id, unit_concept_id,
               measurement_source_value, measurement_source_concept_id, value_source_value,
               measurement_event_id, meas_event_field_concept_id)
           VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)""", rows)
    return len(rows)


def etl_dicom_image_features(conn):
    """One IMAGE_FEATURE row per DICOM-attribute MEASUREMENT row (type 32817)."""
    conn.execute("DELETE FROM image_feature WHERE image_feature_type_concept_id = ?", (CONCEPT_TYPE_EHR,))
    fid = conn.execute("SELECT COALESCE(MAX(image_feature_id), 0) FROM image_feature").fetchone()[0]
    conn.execute(
        """INSERT INTO image_feature (image_feature_id, person_id, image_occurrence_id,
               image_feature_event_field_concept_id, image_feature_event_id, image_feature_concept_id,
               image_feature_type_concept_id, image_finding_concept_id, image_finding_id,
               anatomic_site_concept_id, alg_system, alg_datetime)
           SELECT ? + ROW_NUMBER() OVER (ORDER BY m.measurement_id), m.person_id, m.measurement_event_id,
                  ?, m.measurement_id, m.measurement_concept_id, m.measurement_type_concept_id,
                  NULL, NULL, io.anatomic_site_concept_id, NULL, NULL
           FROM measurement m
           JOIN image_occurrence io ON io.image_occurrence_id = m.measurement_event_id
           WHERE m.measurement_type_concept_id = ?""",
        (fid, EVENT_FIELD_MEASUREMENT, CONCEPT_TYPE_EHR))
    return conn.execute("SELECT COUNT(*) FROM image_feature WHERE image_feature_type_concept_id = ?",
                        (CONCEPT_TYPE_EHR,)).fetchone()[0]


DQ_CHECKS_PART1 = [
    # (category, check, SQL returning the number of violating rows)
    ("completeness", "image_occurrence without person / procedure",
     "SELECT COUNT(*) FROM image_occurrence WHERE person_id IS NULL OR procedure_occurrence_id IS NULL"),
    ("completeness", "image_occurrence missing study/series UID or date",
     "SELECT COUNT(*) FROM image_occurrence WHERE image_study_uid IS NULL OR image_series_uid IS NULL OR image_occurrence_date IS NULL"),
    ("completeness", "image_occurrence with modality_concept_id = 0",
     "SELECT COUNT(*) FROM image_occurrence WHERE modality_concept_id = 0"),
    ("completeness", "image_occurrence without wadors_uri AND local_path",
     "SELECT COUNT(*) FROM image_occurrence WHERE wadors_uri IS NULL AND local_path IS NULL"),
    ("completeness", "DICOM measurement without measurement_event_id",
     "SELECT COUNT(*) FROM measurement WHERE measurement_type_concept_id = 32817 AND measurement_event_id IS NULL"),
    ("conformance", "concept_ids not present in CONCEPT (image_occurrence)",
     """SELECT COUNT(*) FROM image_occurrence io LEFT JOIN concept c ON c.concept_id = io.modality_concept_id
        LEFT JOIN concept a ON a.concept_id = io.anatomic_site_concept_id
        WHERE c.concept_id IS NULL OR (io.anatomic_site_concept_id IS NOT NULL AND a.concept_id IS NULL)"""),
    ("conformance", "measurement_concept_id not a DICOM attribute (DICOM rows)",
     """SELECT COUNT(*) FROM measurement m LEFT JOIN concept c ON c.concept_id = m.measurement_concept_id
        WHERE m.measurement_type_concept_id = 32817 AND (c.concept_class_id IS NULL OR c.concept_class_id <> 'DICOM Attributes')"""),
    ("conformance", "measurement without source value",
     "SELECT COUNT(*) FROM measurement WHERE measurement_source_value IS NULL"),
    ("conformance", "CS value stored as number / numeric VR stored as concept",
     """SELECT COUNT(*) FROM measurement WHERE value_as_number IS NOT NULL AND value_as_concept_id IS NOT NULL"""),
    ("conformance", "duplicate (study_uid, series_uid)",
     "SELECT COUNT(*) FROM (SELECT 1 FROM image_occurrence GROUP BY image_study_uid, image_series_uid HAVING COUNT(*) > 1)"),
    ("plausibility", "Pixel Spacing outside 0.05–1.0 mm",
     """SELECT COUNT(*) FROM measurement m JOIN concept c ON c.concept_id = m.measurement_concept_id
        WHERE c.concept_code = '00280030' AND (m.value_as_number < 0.05 OR m.value_as_number > 1.0)"""),
    ("plausibility", "Rows/Columns outside 256–5000 px",
     """SELECT COUNT(*) FROM measurement m JOIN concept c ON c.concept_id = m.measurement_concept_id
        WHERE c.concept_code IN ('00280010','00280011') AND (m.value_as_number < 256 OR m.value_as_number > 5000)"""),
    ("linkage", "measurement_event_id not resolving to image_occurrence",
     """SELECT COUNT(*) FROM measurement m LEFT JOIN image_occurrence io ON io.image_occurrence_id = m.measurement_event_id
        WHERE m.measurement_event_id IS NOT NULL AND io.image_occurrence_id IS NULL"""),
    ("linkage", "measurement.person_id differs from its image's person_id",
     """SELECT COUNT(*) FROM measurement m JOIN image_occurrence io ON io.image_occurrence_id = m.measurement_event_id
        WHERE m.person_id <> io.person_id"""),
    ("linkage", "image person differs from its procedure's person",
     """SELECT COUNT(*) FROM image_occurrence io JOIN procedure_occurrence po USING (procedure_occurrence_id)
        WHERE io.person_id <> po.person_id"""),
]


def run_dq(checks):
    with connect() as conn:
        res = [(cat, name, conn.execute(sql).fetchone()[0]) for cat, name, sql in checks]
        fk = len(conn.execute("PRAGMA foreign_key_check").fetchall())
    res.append(("linkage", "PRAGMA foreign_key_check", fk))
    out = pd.DataFrame(res, columns=["category", "check", "violations"])
    out["status"] = out["violations"].map(lambda n: "PASS" if n == 0 else "REVIEW")
    return out


def build_part1(verbose=True):
    """Run notebook 01 end to end (catch-up)."""
    with connect() as conn:
        create_cdm_tables(conn)
        create_micdm_tables(conn)
        load_dicom_vocabulary(conn, download_vocabulary())
    download_rsna_subset()
    inv = index_dicom_archive()
    flat = extract_series_metadata(inv)
    with connect() as conn:
        inv.to_sql("stg_dicom_inventory", conn, if_exists="replace", index=False)
        flat.to_sql("stg_dicom_metadata_long", conn, if_exists="replace", index=False)
        selected = select_attributes(map_tags_to_concepts(flat, conn))
        etl_person(conn)
        etl_procedure_occurrence(conn)
        etl_image_occurrence(conn)
        n = etl_dicom_measurements(conn, selected)
        etl_dicom_image_features(conn)
    if verbose:
        print(f"notebook 01 rebuilt: {len(inv)} files, {flat.SeriesUID.nunique()} series, {n} DICOM measurements")
'''
open("micdm_part1.py", "w").write(MICDM_PART1_SRC)
import micdm_part1; importlib.reload(micdm_part1)
from micdm_part1 import *

def _ready():
    try:
        return os.path.exists(DB_PATH) and q("""SELECT COUNT(*) FROM measurement WHERE measurement_type_concept_id = 32817""").iloc[0, 0] > 0
    except Exception:
        return False

if _ready():
    print("Earlier notebook output found:", DB_PATH)
else:
    print("Rebuilding earlier notebook output (notebook 01: download + ETL, ~3 min) ...")
    build_part1()

Earlier notebook output found: /content/micdm_tutorial/omop_micdm.db


In [ ]:
q("""SELECT COUNT(*) AS image_occurrence_rows,
            SUM(local_path IS NOT NULL) AS with_local_path,
            COUNT(DISTINCT person_id) AS persons
     FROM image_occurrence""")

,image_occurrence_rows,with_local_path,persons
0,100,100,100


## 1. Wiki step 7 — the algorithm

| | |
|---|---|
| Library | [TorchXRayVision](https://github.com/mlmed/torchxrayvision) (Apache-2.0) — Cohen JP *et al.*, MIDL 2022 |
| Classifier | `DenseNet(weights="densenet121-res224-all")` — 18 findings, ~28 MB |
| Segmenter | `PSPNet()` (ChestX-Det) — 14 anatomical masks, ~270 MB; gives heart and lung geometry |

Why these: they are small, open, run on a free Colab runtime, and produce both kinds of output the ETL guide describes — **coded findings** (classifier) and **quantitative features** (segmentation → cardiothoracic ratio).

One detail that matters for the numbers: for these weights, TorchXRayVision **calibrates** each output so that the published operating point sits at **0.5**. The stored value is that calibrated score, and the yes/no call is `score ≥ 0.5`.

**Concepts used in the next cell**

*Standard / OMOP-defined concepts* (verified 2026-09-29; links open Athena)

| concept_id | Name | Domain | Vocabulary | Class | Code |
|---|---|---|---|---|---|
| [32880](https://athena.ohdsi.org/search-terms/terms/32880) | Standard algorithm | Type Concept | Type Concept | Type Concept | OMOP4976953 |

In [ ]:
import sys, time, uuid, hashlib, platform
import numpy as np

BATCH_SIZE = 8                     # lower to 2 if a GPU runs out of memory
RUN_SEGMENTATION = True            # False skips the 270 MB segmentation model
CLS_WEIGHTS = "densenet121-res224-all"
CLS_INPUT_SIZE, SEG_INPUT_SIZE = 224, 512
CALL_THRESHOLD = 0.5               # operating point after TorchXRayVision calibration
SEG_MASK_THRESHOLD = 0.5
CONCEPT_TYPE_ALGORITHM = 32880     # Type Concept 'Standard algorithm' (as in DICOM2OMOP)


def device():
    import torch
    return "cuda" if torch.cuda.is_available() else "cpu"
# ▶ run
print("device:", device(), "| batch size:", BATCH_SIZE, "| segmentation:", RUN_SEGMENTATION)

device: cuda | batch size: 8 | segmentation: True


## 2. Wiki steps 9 and 11 — concepts for the features

The ETL guide separates **what was found** from **what was measured**:

| IMAGE_FEATURE / OBSERVATION field | Meaning | Here |
|---|---|---|
| `image_finding_concept_id` | the finding ("what was identified?") | SNOMED: *Pneumonia* `255848`, *Pleural effusion* `254061`, … |
| `image_feature_concept_id` = `observation_concept_id` | the feature | *Cardiothoracic ratio* `44807121`; model score and yes/no call have no concept → `0` |
| `anatomic_site_concept_id` | where, more specific than the series | *Lung structure*, *Heart structure*, … |
| `image_feature_type_concept_id` | how it was generated | `32880` Standard algorithm |

**Step 9 — standard concepts first.** Findings map to SNOMED. The *cardiothoracic ratio* has a standard SNOMED concept (`44807121`). Its domain is *Measurement*; it is stored in OBSERVATION together with the other algorithm outputs (tutorial decision, see section 4).

**Step 11 — where no concept exists, none is invented here.** There is no standard concept for "algorithm score for finding X", nor for segmented lung/heart area. Those rows get `observation_concept_id = 0` and `image_feature_concept_id = 0`, and what they are is kept in `observation_source_value` (e.g. `Pneumonia score`, `total_lung_area_cm2`). The finding itself is still coded (`image_finding_concept_id`). Whether the WG should define shared custom concepts for these is an open item.

Mappings marked *approximate* are editorial choices, printed so a reviewer can challenge them.

**Concepts used in the next cell**

*Standard / OMOP-defined concepts* (verified 2026-09-29; links open Athena)

| concept_id | Name | Domain | Vocabulary | Class | Code |
|---|---|---|---|---|---|
| [0](https://athena.ohdsi.org/search-terms/terms/0) | No matching concept | Metadata | None | Undefined | No matching concept |
| [4188539](https://athena.ohdsi.org/search-terms/terms/4188539) | Yes | Meas Value | SNOMED | Qualifier Value | 373066001 |
| [4188540](https://athena.ohdsi.org/search-terms/terms/4188540) | No | Meas Value | SNOMED | Qualifier Value | 373067005 |
| [44807121](https://athena.ohdsi.org/search-terms/terms/44807121) | Cardiothoracic ratio | Measurement | SNOMED | Observable Entity | 815571000000107 |
| [9483](https://athena.ohdsi.org/search-terms/terms/9483) | square centimeter | Unit | UCUM | Unit | cm2 |
| [8523](https://athena.ohdsi.org/search-terms/terms/8523) | ratio | Unit | UCUM | Unit | {ratio} |
| [4184181](https://athena.ohdsi.org/search-terms/terms/4184181) | Thoracic cavity structure | Spec Anatomic Site | SNOMED | Body Structure | 43799004 |
| [4213162](https://athena.ohdsi.org/search-terms/terms/4213162) | Lung structure | Spec Anatomic Site | SNOMED | Body Structure | 39607008 |
| [4217142](https://athena.ohdsi.org/search-terms/terms/4217142) | Heart structure | Spec Anatomic Site | SNOMED | Body Structure | 80891009 |
| [32880](https://athena.ohdsi.org/search-terms/terms/32880) | Standard algorithm | Type Concept | Type Concept | Type Concept | OMOP4976953 |
| [1147304](https://athena.ohdsi.org/search-terms/terms/1147304) | observation | Metadata | CDM | Table | CDM360 |
| [261880](https://athena.ohdsi.org/search-terms/terms/261880) | Atelectasis | Condition | SNOMED | Disorder | 46621007 |
| [4319884](https://athena.ohdsi.org/search-terms/terms/4319884) | Consolidation | Observation | SNOMED | Morph Abnormality | 9656002 |
| [4264333](https://athena.ohdsi.org/search-terms/terms/4264333) | Infiltration | Observation | SNOMED | Morph Abnormality | 47351003 |
| [253796](https://athena.ohdsi.org/search-terms/terms/253796) | Pneumothorax | Condition | SNOMED | Disorder | 36118008 |
| [4078925](https://athena.ohdsi.org/search-terms/terms/4078925) | Pulmonary edema | Condition | SNOMED | Disorder | 19242006 |
| [4169883](https://athena.ohdsi.org/search-terms/terms/4169883) | Emphysema | Observation | SNOMED | Morph Abnormality | 49158009 |
| [4197819](https://athena.ohdsi.org/search-terms/terms/4197819) | Fibrosis of lung | Condition | SNOMED | Disorder | 51615001 |
| [254061](https://athena.ohdsi.org/search-terms/terms/254061) | Pleural effusion | Condition | SNOMED | Disorder | 60046008 |
| [255848](https://athena.ohdsi.org/search-terms/terms/255848) | Pneumonia | Condition | SNOMED | Disorder | 233604007 |
| [4249320](https://athena.ohdsi.org/search-terms/terms/4249320) | Thickening of pleura | Condition | SNOMED | Disorder | 73725006 |
| [314658](https://athena.ohdsi.org/search-terms/terms/314658) | Cardiomegaly | Condition | SNOMED | Disorder | 8186001 |
| [4101098](https://athena.ohdsi.org/search-terms/terms/4101098) | Nodule | Observation | SNOMED | Morph Abnormality | 27925004 |
| [4214641](https://athena.ohdsi.org/search-terms/terms/4214641) | Mass | Observation | SNOMED | Morph Abnormality | 4147007 |
| [4190388](https://athena.ohdsi.org/search-terms/terms/4190388) | Hernia | Observation | SNOMED | Morph Abnormality | 414403008 |
| [4116778](https://athena.ohdsi.org/search-terms/terms/4116778) | Lesion of lung | Condition | SNOMED | Disorder | 301232003 |
| [4217808](https://athena.ohdsi.org/search-terms/terms/4217808) | Fracture | Observation | SNOMED | Morph Abnormality | 72704001 |
| [4178364](https://athena.ohdsi.org/search-terms/terms/4178364) | Widened mediastinum | Condition | SNOMED | Clinical Finding | 363646005 |

In [ ]:
CONCEPT_YES, CONCEPT_NO = 4188539, 4188540          # SNOMED 'Yes' / 'No' (Meas Value)
CONCEPT_CTR = 44807121                               # SNOMED 'Cardiothoracic ratio'
CONCEPT_UNIT_CM2, CONCEPT_UNIT_RATIO = 9483, 8523
EVENT_FIELD_OBSERVATION = 1147304                    # CDM 'observation' table: image_feature -> OBSERVATION
SITE_THORAX, SITE_LUNG, SITE_HEART = 4184181, 4213162, 4217142

# label -> (SNOMED finding concept, name, mapping status)   [labels as in model.pathologies]
PATHOLOGY_CONCEPT = {
    "Atelectasis": (261880, "Atelectasis", "exact"),
    "Consolidation": (4319884, "Consolidation", "exact"),
    "Infiltration": (4264333, "Infiltration", "exact"),
    "Pneumothorax": (253796, "Pneumothorax", "exact"),
    "Edema": (4078925, "Pulmonary edema", "approximate"),
    "Emphysema": (4169883, "Emphysema", "exact"),
    "Fibrosis": (4197819, "Fibrosis of lung", "approximate"),
    "Effusion": (254061, "Pleural effusion", "approximate"),
    "Pneumonia": (255848, "Pneumonia", "exact"),
    "Pleural_Thickening": (4249320, "Thickening of pleura", "approximate"),
    "Cardiomegaly": (314658, "Cardiomegaly", "exact"),
    "Nodule": (4101098, "Nodule", "exact"),
    "Mass": (4214641, "Mass", "exact"),
    "Hernia": (4190388, "Hernia", "exact"),
    "Lung Lesion": (4116778, "Lesion of lung", "approximate"),
    "Fracture": (4217808, "Fracture", "exact"),
    "Lung Opacity": (0, "(no standard concept)", "unmapped"),
    "Enlarged Cardiomediastinum": (4178364, "Widened mediastinum", "approximate"),
}
PATHOLOGIES = list(PATHOLOGY_CONCEPT)
PATHOLOGY_SITE = {p: SITE_LUNG for p in PATHOLOGIES}
PATHOLOGY_SITE.update({"Pneumothorax": SITE_THORAX, "Effusion": SITE_THORAX, "Pleural_Thickening": SITE_THORAX,
                       "Enlarged Cardiomediastinum": SITE_THORAX, "Cardiomegaly": SITE_HEART,
                       "Nodule": None, "Mass": None, "Fracture": None, "Hernia": None})

# segmentation outputs: key -> (observation_concept_id, unit, site). Only the CTR has a standard
# concept; the areas are unmapped (0) and identified by observation_source_value = key.
SEG_FEATURES = {
    "cardiothoracic_ratio": (CONCEPT_CTR, CONCEPT_UNIT_RATIO, SITE_HEART),
    "total_lung_area_cm2": (CONCEPT_UNMAPPED, CONCEPT_UNIT_CM2, SITE_LUNG),
    "heart_area_cm2": (CONCEPT_UNMAPPED, CONCEPT_UNIT_CM2, SITE_HEART),
}

STANDARD_CONCEPTS_PART2 = [
    (32880, "Standard algorithm", "Type Concept", "Type Concept", "Type Concept", "S", "OMOP4976953"),
    (1147304, "observation", "Metadata", "CDM", "Table", "S", "CDM360"),
    (4188539, "Yes", "Meas Value", "SNOMED", "Qualifier Value", "S", "373066001"),
    (4188540, "No", "Meas Value", "SNOMED", "Qualifier Value", "S", "373067005"),
    (44807121, "Cardiothoracic ratio", "Measurement", "SNOMED", "Observable Entity", "S", "815571000000107"),
    (9483, "square centimeter", "Unit", "UCUM", "Unit", "S", "cm2"),
    (8523, "ratio", "Unit", "UCUM", "Unit", "S", "{ratio}"),
    (4217142, "Heart structure", "Spec Anatomic Site", "SNOMED", "Body Structure", "S", "80891009"),
] + [   # SNOMED finding concepts (domain/class as in the vocabulary)
    (261880, "Atelectasis", "Condition", "SNOMED", "Disorder", "S", "46621007"),
    (4319884, "Consolidation", "Observation", "SNOMED", "Morph Abnormality", "S", "9656002"),
    (4264333, "Infiltration", "Observation", "SNOMED", "Morph Abnormality", "S", "47351003"),
    (253796, "Pneumothorax", "Condition", "SNOMED", "Disorder", "S", "36118008"),
    (4078925, "Pulmonary edema", "Condition", "SNOMED", "Disorder", "S", "19242006"),
    (4169883, "Emphysema", "Observation", "SNOMED", "Morph Abnormality", "S", "49158009"),
    (4197819, "Fibrosis of lung", "Condition", "SNOMED", "Disorder", "S", "51615001"),
    (254061, "Pleural effusion", "Condition", "SNOMED", "Disorder", "S", "60046008"),
    (255848, "Pneumonia", "Condition", "SNOMED", "Disorder", "S", "233604007"),
    (4249320, "Thickening of pleura", "Condition", "SNOMED", "Disorder", "S", "73725006"),
    (314658, "Cardiomegaly", "Condition", "SNOMED", "Disorder", "S", "8186001"),
    (4101098, "Nodule", "Observation", "SNOMED", "Morph Abnormality", "S", "27925004"),
    (4214641, "Mass", "Observation", "SNOMED", "Morph Abnormality", "S", "4147007"),
    (4190388, "Hernia", "Observation", "SNOMED", "Morph Abnormality", "S", "414403008"),
    (4116778, "Lesion of lung", "Condition", "SNOMED", "Disorder", "S", "301232003"),
    (4217808, "Fracture", "Observation", "SNOMED", "Morph Abnormality", "S", "72704001"),
    (4178364, "Widened mediastinum", "Condition", "SNOMED", "Clinical Finding", "S", "363646005"),
]


def define_feature_concepts(conn):
    """Adds the standard concepts used below to the display subset. No custom concepts are created."""
    insert_concepts(conn, STANDARD_CONCEPTS_PART2)
    return len(STANDARD_CONCEPTS_PART2)
# ▶ run
with connect() as conn:
    n_std = define_feature_concepts(conn)
print(f"{n_std} standard concepts added to the display subset; no custom concepts created\n")
pd.DataFrame([(p, c, n, s) for p, (c, n, s) in PATHOLOGY_CONCEPT.items()],
             columns=["model label", "finding concept_id", "SNOMED name", "mapping"]).query("mapping != 'exact'")

25 standard concepts added to the display subset; no custom concepts created



,model label,finding concept_id,SNOMED name,mapping
4,Edema,4078925,Pulmonary edema,approximate
6,Fibrosis,4197819,Fibrosis of lung,approximate
7,Effusion,254061,Pleural effusion,approximate
9,Pleural_Thickening,4249320,Thickening of pleura,approximate
14,Lung Lesion,4116778,Lesion of lung,approximate
16,Lung Opacity,0,(no standard concept),unmapped
17,Enlarged Cardiomediastinum,4178364,Widened mediastinum,approximate


## 3. Wiki step 10a — run the algorithm

Steps: read `IMAGE_OCCURRENCE.local_path` → decode pixels → model. Every preprocessing choice changes the numbers, so each is written into the provenance record:

1. decode (`pylibjpeg` handles RSNA's JPEG Baseline), 2. invert if `MONOCHROME1`, 3. scale to [-1024, 1024] using `BitsStored`, 4. centre-crop and resize (224 for the classifier, 512 for the segmenter).

The SHA-256 of each weight file is recorded too: "DenseNet-121" is a name, the checksum identifies the exact tensor that produced the number.

In [ ]:
PREPROCESSING = ("pydicom pixel_array; invert if MONOCHROME1; xrv.datasets.normalize(arr, 2**BitsStored-1); "
                 "XRayCenterCrop; XRayResizer(size, engine='skimage')")


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def read_cxr(path):
    import pydicom, torchxrayvision as xrv
    ds = pydicom.dcmread(path)
    arr = ds.pixel_array.astype(np.float32)
    if str(ds.get("PhotometricInterpretation", "")).upper() == "MONOCHROME1":
        arr = arr.max() - arr
    if arr.ndim == 3:
        arr = arr.mean(axis=2)
    bits = int(ds.get("BitsStored", 0) or 0)
    maxval = max(float(2 ** bits - 1) if bits else float(arr.max()), float(arr.max()))
    img = xrv.datasets.normalize(arr, maxval)[None, ...]
    sp = ds.get("PixelSpacing")
    meta = {"rows": arr.shape[0], "columns": arr.shape[1],
            "spacing": (float(sp[0]), float(sp[1])) if sp else None}
    return img, meta


def model_input(img, size):
    import torchxrayvision as xrv
    img = xrv.datasets.XRayCenterCrop()(img)
    return xrv.datasets.XRayResizer(size, engine="skimage")(img).astype(np.float32)


def image_rows():
    return q("""SELECT image_occurrence_id, person_id, image_occurrence_date, local_path
                FROM image_occurrence WHERE local_path IS NOT NULL ORDER BY image_occurrence_id""")


def run_record(task, model, model_name, version, hyper):
    import torch, torchxrayvision as xrv
    wfile = getattr(model, "weights_filename_local", None)
    sha = sha256_file(wfile) if wfile and os.path.exists(wfile) else None
    rec = {"run_uuid": str(uuid.uuid4()), "task": task, "model_name": model_name, "model_version": version,
           "library": f"torchxrayvision {xrv.__version__}", "weights_sha256": sha,
           "hyperparameters": json.dumps(hyper), "preprocessing": PREPROCESSING,
           "software_env": json.dumps({"python": sys.version.split()[0], "torch": torch.__version__,
                                       "platform": platform.platform(), "device": device()}),
           "run_datetime": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")}
    # MI-CDM's own provenance field: one self-describing string per algorithm run
    rec["alg_system"] = f"{model_name} [{version}] {rec['library']} sha256:{(sha or 'n/a')[:12]} run:{rec['run_uuid'][:8]}"
    return rec


def run_classification(rows):
    import torch, torchxrayvision as xrv
    model = xrv.models.DenseNet(weights=CLS_WEIGHTS).to(device()).eval()
    assert list(model.pathologies) == PATHOLOGIES or set(model.pathologies) == set(PATHOLOGIES), \
        "TorchXRayVision label set changed — update PATHOLOGY_CONCEPT"
    labels = list(model.pathologies)
    out, t0 = [], time.time()
    for i in range(0, len(rows), BATCH_SIZE):
        chunk = rows.iloc[i:i + BATCH_SIZE]
        x = np.stack([model_input(read_cxr(p)[0], CLS_INPUT_SIZE) for p in chunk.local_path])
        with torch.no_grad():
            scores = model(torch.from_numpy(x).to(device())).cpu().numpy()
        for ioid, s in zip(chunk.image_occurrence_id, scores):
            out.append({"image_occurrence_id": ioid, **dict(zip(labels, map(float, s)))})
        print(f"  classified {min(i + BATCH_SIZE, len(rows))}/{len(rows)}  ({time.time() - t0:.0f}s)")
    rec = run_record("classification", model, "TorchXRayVision DenseNet121", CLS_WEIGHTS,
                     {"input_size": CLS_INPUT_SIZE, "call_threshold": CALL_THRESHOLD,
                      "calibration": "xrv op_norm (operating point -> 0.5)",
                      "op_threshs": [None if np.isnan(v) else float(v) for v in model.op_threshs.cpu().numpy()]})
    return pd.DataFrame(out), rec


def _max_width(mask):
    """Widest left-to-right extent of a boolean mask (in grid columns)."""
    if not mask.any():
        return 0.0
    cols = np.arange(mask.shape[1])[None, :]
    right = np.where(mask, cols, -1).max(axis=1)
    left = np.where(mask, cols, mask.shape[1]).min(axis=1)
    return float(np.where(mask.any(axis=1), right - left + 1, 0).max())


def run_segmentation(rows):
    import torch, torchxrayvision as xrv
    model = xrv.baseline_models.chestx_det.PSPNet().to(device()).eval()
    t = {name: i for i, name in enumerate(model.targets)}
    out, t0 = [], time.time()
    for i in range(0, len(rows), 4):
        chunk = rows.iloc[i:i + 4]
        loaded = [read_cxr(p) for p in chunk.local_path]
        x = np.stack([model_input(img, SEG_INPUT_SIZE) for img, _ in loaded])
        with torch.no_grad():
            masks = torch.sigmoid(model(torch.from_numpy(x).to(device()))).cpu().numpy() > SEG_MASK_THRESHOLD
        for ioid, m, (_, meta) in zip(chunk.image_occurrence_id, masks, loaded):
            lungs, heart = m[t["Left Lung"]] | m[t["Right Lung"]], m[t["Heart"]]
            lw, hw = _max_width(lungs), _max_width(heart)
            f = {"image_occurrence_id": ioid, "cardiothoracic_ratio": round(hw / lw, 4) if lw else None}
            if meta["spacing"]:
                cm2 = (min(meta["rows"], meta["columns"]) / SEG_INPUT_SIZE) ** 2 * meta["spacing"][0] * meta["spacing"][1] / 100
                f["total_lung_area_cm2"] = round(float(lungs.sum()) * cm2, 2)
                f["heart_area_cm2"] = round(float(heart.sum()) * cm2, 2)
            out.append(f)
        print(f"  segmented {min(i + 4, len(rows))}/{len(rows)}  ({time.time() - t0:.0f}s)")
    rec = run_record("segmentation", model, "TorchXRayVision PSPNet (ChestX-Det)", "chestx_det PSPNet",
                     {"input_size": SEG_INPUT_SIZE, "mask_threshold": SEG_MASK_THRESHOLD,
                      "ctr_definition": "max transverse heart-mask width / max transverse lung-mask width",
                      "area_definition": "mask pixels x (crop/512)^2 x PixelSpacing_row x PixelSpacing_col"})
    return pd.DataFrame(out), rec
# ▶ run
rows = image_rows()
cls_df, cls_run = run_classification(rows)
print("\nclassifier provenance:", cls_run["alg_system"])
cls_df.round(3).head()

  classified 8/100  (1s)
  classified 16/100  (1s)
  classified 24/100  (2s)
  classified 32/100  (3s)
  classified 40/100  (4s)
  classified 48/100  (5s)
  classified 56/100  (8s)
  classified 64/100  (9s)
  classified 72/100  (10s)
  classified 80/100  (10s)
  classified 88/100  (11s)
  classified 96/100  (11s)
  classified 100/100  (11s)

classifier provenance: TorchXRayVision DenseNet121 [densenet121-res224-all] torchxrayvision 1.5.5 sha256:56524913dd16 run:b7e816d0


,image_occurrence_id,Atelectasis,Consolidation,Infiltration,Pneumothorax,Edema,Emphysema,Fibrosis,Effusion,Pneumonia,Pleural_Thickening,Cardiomegaly,Nodule,Mass,Hernia,Lung Lesion,Fracture,Lung Opacity,Enlarged Cardiomediastinum
0,1,0.596,0.519,0.529,0.505,0.011,0.221,0.542,0.553,0.017,0.539,0.066,0.548,0.618,0.022,0.158,0.554,0.630,0.339
1,2,0.564,0.503,0.478,0.507,0.138,0.504,0.503,0.150,0.487,0.386,0.022,0.523,0.605,0.006,0.104,0.521,0.516,0.525
2,3,0.248,0.145,0.514,0.239,0.014,0.197,0.515,0.069,0.055,0.242,0.028,0.345,0.214,0.034,0.021,0.252,0.289,0.126
3,4,0.347,0.151,0.555,0.184,0.002,0.278,0.517,0.068,0.010,0.296,0.002,0.466,0.257,0.019,0.062,0.306,0.123,0.067
4,5,0.084,0.188,0.521,0.506,0.004,0.315,0.503,0.063,0.014,0.466,0.002,0.500,0.512,0.002,0.016,0.457,0.361,0.090


In [ ]:
if RUN_SEGMENTATION:
    seg_df, seg_run = run_segmentation(rows)
    print("\nsegmenter provenance:", seg_run["alg_system"])
    display(seg_df.head())
else:
    seg_df, seg_run = None, None

  segmented 4/100  (1s)
  segmented 8/100  (2s)
  segmented 12/100  (4s)
  segmented 16/100  (5s)
  segmented 20/100  (6s)
  segmented 24/100  (8s)
  segmented 28/100  (10s)
  segmented 32/100  (11s)
  segmented 36/100  (12s)
  segmented 40/100  (12s)
  segmented 44/100  (13s)
  segmented 48/100  (14s)
  segmented 52/100  (15s)
  segmented 56/100  (16s)
  segmented 60/100  (17s)
  segmented 64/100  (18s)
  segmented 68/100  (19s)
  segmented 72/100  (20s)
  segmented 76/100  (21s)
  segmented 80/100  (22s)
  segmented 84/100  (23s)
  segmented 88/100  (24s)
  segmented 92/100  (24s)
  segmented 96/100  (25s)
  segmented 100/100  (26s)

segmenter provenance: TorchXRayVision PSPNet (ChestX-Det) [chestx_det PSPNet] torchxrayvision 1.5.5 sha256:019b167eac6b run:3d5b126d


,image_occurrence_id,cardiothoracic_ratio,total_lung_area_cm2,heart_area_cm2
0,1,0.4031,52.77,12.40
1,2,0.4671,75.02,17.81
2,3,0.4467,39.93,9.09
3,4,0.4081,71.02,13.03
4,5,0.4298,63.08,12.66


## 4. Wiki step 10b — load features into IMAGE_FEATURE + OBSERVATION

This mirrors DICOM2OMOP `transform_load_imaging_algo_results.ipynb`: melt the wide result table to long, join `IMAGE_OCCURRENCE` on the series, continue `observation_id` / `image_feature_id` from the current maximum, then insert.

**Decision: the algorithm values go to OBSERVATION, not MEASUREMENT.** The CDM keeps MEASUREMENT for results of a standardized test; a model score or call is not one. OBSERVATION holds attribute–value facts (`value_as_number`, `value_as_concept_id`), and the CDM recommends *Yes* (`4188539`) for positive assertions. Exception to note: the CTR concept `44807121` belongs to the Measurement domain, which the CDM would route to MEASUREMENT; it is kept in OBSERVATION here so that all algorithm output sits in one table. The DICOM header attributes from notebook 01 stay in MEASUREMENT.

For every *(image, finding)* the classifier yields **two** OBSERVATION rows — the score (`value_as_number`) and the call (`value_as_concept_id` = Yes/No) — and **two** IMAGE_FEATURE rows pointing at them. Both share one `image_finding_id` (ETL guide §6.4: the grouper for "these features describe the same finding"). Thresholding loses information, so the score is kept.

| Field | Value | Guide |
|---|---|---|
| `image_feature_event_field_concept_id` | `1147304` (OBSERVATION) | §7 |
| `image_feature_event_id` | the `observation_id` | §7 |
| `observation_event_id` / `obs_event_field_concept_id` | `image_occurrence_id` / `0` (no Field concept yet) | §7 |
| `image_feature_type_concept_id`, `observation_type_concept_id` | `32880` Standard algorithm | §6.6 |
| `alg_system`, `alg_datetime` | model + version + weight checksum + run id; run time | §6.7 |

Detail the two MI-CDM columns cannot hold (thresholds, preprocessing, library versions) goes into a **site-local** `local_model_run` table, which §6.7 explicitly allows. It is not part of OMOP or MI-CDM.

**Concepts used in the next cell**

*Standard / OMOP-defined concepts* (verified 2026-09-29; links open Athena)

| concept_id | Name | Domain | Vocabulary | Class | Code |
|---|---|---|---|---|---|
| [0](https://athena.ohdsi.org/search-terms/terms/0) | No matching concept | Metadata | None | Undefined | No matching concept |
| [32880](https://athena.ohdsi.org/search-terms/terms/32880) | Standard algorithm | Type Concept | Type Concept | Type Concept | OMOP4976953 |
| [1147304](https://athena.ohdsi.org/search-terms/terms/1147304) | observation | Metadata | CDM | Table | CDM360 |
| [4188539](https://athena.ohdsi.org/search-terms/terms/4188539) | Yes | Meas Value | SNOMED | Qualifier Value | 373066001 |
| [4188540](https://athena.ohdsi.org/search-terms/terms/4188540) | No | Meas Value | SNOMED | Qualifier Value | 373067005 |
| [44807121](https://athena.ohdsi.org/search-terms/terms/44807121) | Cardiothoracic ratio | Measurement | SNOMED | Observable Entity | 815571000000107 |
| [8523](https://athena.ohdsi.org/search-terms/terms/8523) | ratio | Unit | UCUM | Unit | {ratio} |
| [9483](https://athena.ohdsi.org/search-terms/terms/9483) | square centimeter | Unit | UCUM | Unit | cm2 |

In [ ]:
OBSERVATION_DDL = """
CREATE TABLE IF NOT EXISTS observation (          -- OMOP CDM v5.4
    observation_id                INTEGER NOT NULL PRIMARY KEY,
    person_id                     INTEGER NOT NULL REFERENCES person (person_id),
    observation_concept_id        INTEGER NOT NULL,
    observation_date              DATE NOT NULL,
    observation_datetime          TIMESTAMP,
    observation_type_concept_id   INTEGER NOT NULL,
    value_as_number               FLOAT,
    value_as_string               VARCHAR(60),
    value_as_concept_id           INTEGER,
    qualifier_concept_id          INTEGER,
    unit_concept_id               INTEGER,
    provider_id                   INTEGER,
    visit_occurrence_id           INTEGER,
    visit_detail_id               INTEGER,
    observation_source_value      VARCHAR(50),
    observation_source_concept_id INTEGER,
    unit_source_value             VARCHAR(50),
    qualifier_source_value        VARCHAR(50),
    value_source_value            VARCHAR(50),
    observation_event_id          INTEGER,
    obs_event_field_concept_id    INTEGER);
"""

LOCAL_MODEL_RUN_DDL = """
CREATE TABLE IF NOT EXISTS local_model_run (      -- SITE-LOCAL, not OMOP / MI-CDM (ETL guide §6.7)
    model_run_id INTEGER PRIMARY KEY, run_uuid TEXT NOT NULL, task TEXT, model_name TEXT,
    model_version TEXT, library TEXT, weights_sha256 TEXT, hyperparameters TEXT,
    preprocessing TEXT, software_env TEXT, run_datetime TEXT, alg_system TEXT NOT NULL);
"""


def etl_image_features(conn, cls_df, cls_run, seg_df=None, seg_run=None):
    # re-runnable: remove only algorithm-derived rows
    conn.execute("DELETE FROM image_feature WHERE image_feature_type_concept_id = ?", (CONCEPT_TYPE_ALGORITHM,))
    conn.executescript(OBSERVATION_DDL)
    conn.execute("DELETE FROM observation WHERE observation_type_concept_id = ?", (CONCEPT_TYPE_ALGORITHM,))
    conn.executescript(LOCAL_MODEL_RUN_DDL)
    conn.execute("DELETE FROM local_model_run")
    runs = [r for r in (cls_run, seg_run) if r]
    cols = ["run_uuid", "task", "model_name", "model_version", "library", "weights_sha256",
            "hyperparameters", "preprocessing", "software_env", "run_datetime", "alg_system"]
    conn.executemany(f"INSERT INTO local_model_run ({', '.join(cols)}) VALUES ({', '.join('?' * len(cols))})",
                     [tuple(r[c] for c in cols) for r in runs])

    io = pd.read_sql_query("SELECT image_occurrence_id, person_id, image_occurrence_date FROM image_occurrence", conn)
    long = cls_df.melt(id_vars="image_occurrence_id", var_name="label", value_name="score").merge(io, on="image_occurrence_id")
    long = long.sort_values(["image_occurrence_id", "label"]).reset_index(drop=True)

    oid = conn.execute("SELECT COALESCE(MAX(observation_id), 0) FROM observation").fetchone()[0] + 1
    fid = conn.execute("SELECT COALESCE(MAX(image_feature_id), 0) FROM image_feature").fetchone()[0] + 1
    finding = 1
    obs, feat = [], []

    def add(r, concept, num, val_concept, unit, source_value, finding_concept, finding_id, site, run):
        nonlocal oid, fid
        obs.append((oid, int(r.person_id), concept, r.image_occurrence_date, CONCEPT_TYPE_ALGORITHM, num, val_concept,
                    unit, str(source_value)[:50], CONCEPT_UNMAPPED, int(r.image_occurrence_id), FIELD_IMAGE_OCCURRENCE_ID))
        feat.append((fid, int(r.person_id), int(r.image_occurrence_id), EVENT_FIELD_OBSERVATION, oid, concept,
                     CONCEPT_TYPE_ALGORITHM, finding_concept, finding_id, site, run["alg_system"], run["run_datetime"]))
        oid += 1
        fid += 1

    for r in long.itertuples():
        f_concept = PATHOLOGY_CONCEPT[r.label][0]
        site = PATHOLOGY_SITE[r.label]
        present = r.score >= CALL_THRESHOLD
        add(r, CONCEPT_UNMAPPED, round(r.score, 6), None, None,
            f"{r.label} score", f_concept, finding, site, cls_run)
        add(r, CONCEPT_UNMAPPED, None, CONCEPT_YES if present else CONCEPT_NO, None,
            f"{r.label} {'>=' if present else '<'} {CALL_THRESHOLD}", f_concept, finding, site, cls_run)
        finding += 1

    if seg_df is not None:
        slong = (seg_df.melt(id_vars="image_occurrence_id", var_name="key", value_name="value")
                 .dropna().merge(io, on="image_occurrence_id").sort_values(["image_occurrence_id", "key"]))
        for ioid, grp in slong.groupby("image_occurrence_id"):
            for r in grp.itertuples():          # one grouper per image: "cardiothoracic geometry"
                mcid, unit, site = SEG_FEATURES[r.key]
                add(r, mcid, float(r.value), None, unit, r.key, None, finding, site, seg_run)
            finding += 1

    conn.executemany(
        """INSERT INTO observation (observation_id, person_id, observation_concept_id, observation_date,
               observation_type_concept_id, value_as_number, value_as_concept_id, unit_concept_id,
               observation_source_value, observation_source_concept_id, observation_event_id,
               obs_event_field_concept_id) VALUES (?,?,?,?,?,?,?,?,?,?,?,?)""", obs)
    conn.executemany(
        """INSERT INTO image_feature (image_feature_id, person_id, image_occurrence_id,
               image_feature_event_field_concept_id, image_feature_event_id, image_feature_concept_id,
               image_feature_type_concept_id, image_finding_concept_id, image_finding_id,
               anatomic_site_concept_id, alg_system, alg_datetime) VALUES (?,?,?,?,?,?,?,?,?,?,?,?)""", feat)
    return len(obs), len(feat)
# ▶ run
with connect() as conn:
    n_obs, n_feat = etl_image_features(conn, cls_df, cls_run, seg_df, seg_run)
print(f"OBSERVATION rows (algorithm): {n_obs}   IMAGE_FEATURE rows: {n_feat}")

print("\nIMAGE_FEATURE (as stored, algorithm rows):")
display(q("SELECT * FROM image_feature WHERE image_feature_type_concept_id = 32880 LIMIT 5"))

print("\nOBSERVATION (as stored, algorithm rows):")
display(q("SELECT * FROM observation WHERE observation_type_concept_id = 32880 LIMIT 5"))

print("\nLOCAL_MODEL_RUN (as stored):")
display(q("SELECT * FROM local_model_run"))

print("\nIMAGE_FEATURE + OBSERVATION with concept names (image 1):")
display(q("""SELECT f.image_feature_id, f.image_occurrence_id, f.image_finding_id,
            f.image_finding_concept_id, fc.concept_name AS finding,
            f.image_feature_concept_id, mc.concept_name AS feature_concept, o.observation_source_value,
            o.value_as_number, o.value_as_concept_id, vc.concept_name AS value_concept,
            o.unit_concept_id, uc.concept_name AS unit
     FROM image_feature f
     JOIN observation o   ON o.observation_id = f.image_feature_event_id AND f.image_feature_event_field_concept_id = 1147304
     LEFT JOIN concept fc ON fc.concept_id = f.image_finding_concept_id
     LEFT JOIN concept mc ON mc.concept_id = f.image_feature_concept_id
     LEFT JOIN concept vc ON vc.concept_id = o.value_as_concept_id
     LEFT JOIN concept uc ON uc.concept_id = o.unit_concept_id
     WHERE f.image_occurrence_id = 1 AND f.image_feature_type_concept_id = 32880
       AND (fc.concept_name IN ('Pneumonia', 'Cardiomegaly') OR f.image_finding_concept_id IS NULL)
     ORDER BY f.image_feature_id"""))

OBSERVATION rows (algorithm): 3899   IMAGE_FEATURE rows: 3899

IMAGE_FEATURE (as stored, algorithm rows):


,image_feature_id,person_id,image_occurrence_id,image_feature_event_field_concept_id,image_feature_event_id,image_feature_concept_id,image_feature_type_concept_id,image_finding_concept_id,image_finding_id,anatomic_site_concept_id,alg_system,alg_datetime
0,1901,12,1,1147304,1,0,32880,261880,1,4213162,TorchXRayVision DenseNet121 [densenet121-res224-all] tor...,2026-09-29 05:00:24
1,1902,12,1,1147304,2,0,32880,261880,1,4213162,TorchXRayVision DenseNet121 [densenet121-res224-all] tor...,2026-09-29 05:00:24
2,1903,12,1,1147304,3,0,32880,314658,2,4217142,TorchXRayVision DenseNet121 [densenet121-res224-all] tor...,2026-09-29 05:00:24
3,1904,12,1,1147304,4,0,32880,314658,2,4217142,TorchXRayVision DenseNet121 [densenet121-res224-all] tor...,2026-09-29 05:00:24
4,1905,12,1,1147304,5,0,32880,4319884,3,4213162,TorchXRayVision DenseNet121 [densenet121-res224-all] tor...,2026-09-29 05:00:24



OBSERVATION (as stored, algorithm rows):


,observation_id,person_id,observation_concept_id,observation_date,observation_datetime,observation_type_concept_id,value_as_number,value_as_string,value_as_concept_id,qualifier_concept_id,...,provider_id,visit_occurrence_id,visit_detail_id,observation_source_value,observation_source_concept_id,unit_source_value,qualifier_source_value,value_source_value,observation_event_id,obs_event_field_concept_id
0,1,12,0,1901-01-01,None,32880,0.595849,None,NaN,None,...,None,None,None,Atelectasis score,0,None,None,None,1,0
1,2,12,0,1901-01-01,None,32880,NaN,None,4188539.0,None,...,None,None,None,Atelectasis >= 0.5,0,None,None,None,1,0
2,3,12,0,1901-01-01,None,32880,0.066095,None,NaN,None,...,None,None,None,Cardiomegaly score,0,None,None,None,1,0
3,4,12,0,1901-01-01,None,32880,NaN,None,4188540.0,None,...,None,None,None,Cardiomegaly < 0.5,0,None,None,None,1,0
4,5,12,0,1901-01-01,None,32880,0.519131,None,NaN,None,...,None,None,None,Consolidation score,0,None,None,None,1,0



LOCAL_MODEL_RUN (as stored):


,model_run_id,run_uuid,task,model_name,model_version,library,weights_sha256,hyperparameters,preprocessing,software_env,run_datetime,alg_system
0,1,b7e816d0-4941-46dd-b0bc-573d2f40ca59,classification,TorchXRayVision DenseNet121,densenet121-res224-all,torchxrayvision 1.5.5,56524913dd16a906422e8d8b66a7a5c46be1d82eb7ac012d8103776f...,"{""input_size"": 224, ""call_threshold"": 0.5, ""calibration""...",pydicom pixel_array; invert if MONOCHROME1; xrv.datasets...,"{""python"": ""3.13.15"", ""torch"": ""2.11.0+cu128"", ""platform...",2026-09-29 05:00:24,TorchXRayVision DenseNet121 [densenet121-res224-all] tor...
1,2,3d5b126d-ec0b-43d1-b75f-739e286af076,segmentation,TorchXRayVision PSPNet (ChestX-Det),chestx_det PSPNet,torchxrayvision 1.5.5,019b167eac6b729fc1bb92bbbc185fc1730aaa65819f4e3fe718186c...,"{""input_size"": 512, ""mask_threshold"": 0.5, ""ctr_definiti...",pydicom pixel_array; invert if MONOCHROME1; xrv.datasets...,"{""python"": ""3.13.15"", ""torch"": ""2.11.0+cu128"", ""platform...",2026-09-29 05:00:52,TorchXRayVision PSPNet (ChestX-Det) [chestx_det PSPNet] ...



IMAGE_FEATURE + OBSERVATION with concept names (image 1):


,image_feature_id,image_occurrence_id,image_finding_id,image_finding_concept_id,finding,image_feature_concept_id,feature_concept,observation_source_value,value_as_number,value_as_concept_id,value_concept,unit_concept_id,unit
0,1903,1,2,314658.0,Cardiomegaly,0,No matching concept,Cardiomegaly score,0.066095,NaN,None,NaN,None
1,1904,1,2,314658.0,Cardiomegaly,0,No matching concept,Cardiomegaly < 0.5,NaN,4188540.0,No,NaN,None
2,1933,1,17,255848.0,Pneumonia,0,No matching concept,Pneumonia score,0.016735,NaN,None,NaN,None
3,1934,1,17,255848.0,Pneumonia,0,No matching concept,Pneumonia < 0.5,NaN,4188540.0,No,NaN,None
4,5501,1,1801,NaN,None,44807121,Cardiothoracic ratio,cardiothoracic_ratio,0.403100,NaN,None,8523.0,ratio
5,5502,1,1801,NaN,None,0,No matching concept,heart_area_cm2,12.400000,NaN,None,9483.0,square centimeter
6,5503,1,1801,NaN,None,0,No matching concept,total_lung_area_cm2,52.770000,NaN,None,9483.0,square centimeter


## 4b. Provenance round trip

The test of provenance: starting from **one number**, can we reconstruct its full history using only the database? Image → series UID and file, procedure, patient, algorithm (with weight checksum), preprocessing and thresholds.

**Concepts used in the next cell**

*Standard / OMOP-defined concepts* (verified 2026-09-29; links open Athena)

| concept_id | Name | Domain | Vocabulary | Class | Code |
|---|---|---|---|---|---|
| [1147304](https://athena.ohdsi.org/search-terms/terms/1147304) | observation | Metadata | CDM | Table | CDM360 |
| [255848](https://athena.ohdsi.org/search-terms/terms/255848) | Pneumonia | Condition | SNOMED | Disorder | 233604007 |
| [4163872](https://athena.ohdsi.org/search-terms/terms/4163872) | Plain X-ray of chest | Procedure | SNOMED | Procedure | 399208008 |

In [ ]:
q("""
SELECT o.observation_id, o.value_as_number AS score, fc.concept_name AS finding,
       io.image_occurrence_id, io.image_series_uid, io.local_path,
       po.procedure_occurrence_id, pc.concept_name AS procedure,
       f.alg_system, f.alg_datetime,
       r.weights_sha256, r.preprocessing, r.hyperparameters
FROM observation o
JOIN image_feature f         ON f.image_feature_event_id = o.observation_id AND f.image_feature_event_field_concept_id = 1147304
JOIN image_occurrence io     ON io.image_occurrence_id = o.observation_event_id
JOIN procedure_occurrence po ON po.procedure_occurrence_id = io.procedure_occurrence_id
LEFT JOIN concept pc         ON pc.concept_id = po.procedure_concept_id
LEFT JOIN concept fc         ON fc.concept_id = f.image_finding_concept_id
LEFT JOIN local_model_run r  ON r.alg_system = f.alg_system
WHERE fc.concept_name = 'Pneumonia' AND o.value_as_number IS NOT NULL
ORDER BY o.value_as_number DESC LIMIT 1
""").T

,0
observation_id,1617
score,0.985003
finding,Pneumonia
image_occurrence_id,45
image_series_uid,1.2.276.0.7230010.3.1.3.8323329.10044.1517874346.509901
local_path,/content/micdm_tutorial/dicom/mdai_rsna_project_x9N20BZa...
procedure_occurrence_id,45
procedure,Plain X-ray of chest
alg_system,TorchXRayVision DenseNet121 [densenet121-res224-all] tor...
alg_datetime,2026-09-29 05:00:24


## 5. Data quality for derived features

**Concepts used in the next cell**

*Standard / OMOP-defined concepts* (verified 2026-09-29; links open Athena)

| concept_id | Name | Domain | Vocabulary | Class | Code |
|---|---|---|---|---|---|
| [0](https://athena.ohdsi.org/search-terms/terms/0) | No matching concept | Metadata | None | Undefined | No matching concept |
| [1147330](https://athena.ohdsi.org/search-terms/terms/1147330) | measurement | Metadata | CDM | Table | CDM386 |
| [1147304](https://athena.ohdsi.org/search-terms/terms/1147304) | observation | Metadata | CDM | Table | CDM360 |
| [32880](https://athena.ohdsi.org/search-terms/terms/32880) | Standard algorithm | Type Concept | Type Concept | Type Concept | OMOP4976953 |
| [44807121](https://athena.ohdsi.org/search-terms/terms/44807121) | Cardiothoracic ratio | Measurement | SNOMED | Observable Entity | 815571000000107 |

In [ ]:
DQ_CHECKS_PART2 = [
    ("linkage", "image_feature -> image_occurrence unresolved",
     "SELECT COUNT(*) FROM image_feature f LEFT JOIN image_occurrence io USING (image_occurrence_id) WHERE io.image_occurrence_id IS NULL"),
    ("linkage", "image_feature -> measurement unresolved",
     """SELECT COUNT(*) FROM image_feature f LEFT JOIN measurement m ON m.measurement_id = f.image_feature_event_id
        WHERE f.image_feature_event_field_concept_id = 1147330 AND m.measurement_id IS NULL"""),
    ("linkage", "feature and its measurement point at different images",
     """SELECT COUNT(*) FROM image_feature f JOIN measurement m ON m.measurement_id = f.image_feature_event_id
        WHERE f.image_feature_event_field_concept_id = 1147330 AND m.measurement_event_id <> f.image_occurrence_id"""),
    ("linkage", "image_feature -> observation unresolved",
     """SELECT COUNT(*) FROM image_feature f LEFT JOIN observation o ON o.observation_id = f.image_feature_event_id
        WHERE f.image_feature_event_field_concept_id = 1147304 AND o.observation_id IS NULL"""),
    ("linkage", "feature and its observation point at different images",
     """SELECT COUNT(*) FROM image_feature f JOIN observation o ON o.observation_id = f.image_feature_event_id
        WHERE f.image_feature_event_field_concept_id = 1147304 AND o.observation_event_id <> f.image_occurrence_id"""),
    ("linkage", "feature person differs from image person",
     "SELECT COUNT(*) FROM image_feature f JOIN image_occurrence io USING (image_occurrence_id) WHERE f.person_id <> io.person_id"),
    ("completeness", "algorithm feature without alg_system / alg_datetime",
     "SELECT COUNT(*) FROM image_feature WHERE image_feature_type_concept_id = 32880 AND (alg_system IS NULL OR alg_datetime IS NULL)"),
    ("completeness", "alg_system not traceable to local_model_run",
     "SELECT COUNT(*) FROM image_feature f LEFT JOIN local_model_run r USING (alg_system) WHERE f.image_feature_type_concept_id = 32880 AND r.model_run_id IS NULL"),
    ("completeness", "model run without weight checksum",
     "SELECT COUNT(*) FROM local_model_run WHERE weights_sha256 IS NULL"),
    ("conformance", "finding concept_id = 0 (unmapped finding)",
     "SELECT COUNT(*) FROM image_feature WHERE image_finding_concept_id = 0"),
    ("plausibility", "classifier score outside [0, 1]",
     """SELECT COUNT(*) FROM observation WHERE observation_type_concept_id = 32880
        AND observation_source_value LIKE '% score' AND (value_as_number < 0 OR value_as_number > 1)"""),
]
# ▶ run
run_dq(DQ_CHECKS_PART1 + DQ_CHECKS_PART2)

,category,check,violations,status
0,completeness,image_occurrence without person / procedure,0,PASS
1,completeness,image_occurrence missing study/series UID or date,0,PASS
2,completeness,image_occurrence with modality_concept_id = 0,0,PASS
3,completeness,image_occurrence without wadors_uri AND local_path,0,PASS
4,completeness,DICOM measurement without measurement_event_id,0,PASS
5,conformance,concept_ids not present in CONCEPT (image_occurrence),0,PASS
6,conformance,measurement_concept_id not a DICOM attribute (DICOM rows),0,PASS
7,conformance,measurement without source value,0,PASS
8,conformance,CS value stored as number / numeric VR stored as concept,0,PASS
9,conformance,"duplicate (study_uid, series_uid)",0,PASS


## Summary

| Table | New rows | Holds |
|---|---|---|
| `image_feature` (type 32880) | 2 per (image × finding) + segmentation features | what/where/which algorithm |
| `observation` (type 32880) | same count | the values |
| `local_model_run` (site-local) | 1 per model run | full recipe |

In [ ]:
# Download the complete database (notebooks 01 + 02) to your computer (and Upload it to SQLite Viewer Web)
print(q("""SELECT 'person' AS table_name, COUNT(*) AS n FROM person
           UNION ALL SELECT 'image_occurrence', COUNT(*) FROM image_occurrence
           UNION ALL SELECT 'measurement (DICOM)', COUNT(*) FROM measurement
           UNION ALL SELECT 'observation (algorithm)', COUNT(*) FROM observation
           UNION ALL SELECT 'image_feature', COUNT(*) FROM image_feature
           UNION ALL SELECT 'local_model_run', COUNT(*) FROM local_model_run""").to_string(index=False))
try:
    from google.colab import files
    files.download(DB_PATH)
except ImportError:
    print("Not on Colab — database is at", DB_PATH)

             table_name    n
                 person  100
       image_occurrence  100
    measurement (DICOM) 5799
observation (algorithm) 3899
          image_feature 5799
        local_model_run    2


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>